# Import Library

In [1]:
import pandas as pd
from pathlib import Path

# Load Data

Baca file raw

In [2]:
file_path = "./data/ground-truth/gt_6m.csv"
# text = Path(file_path).read_text(encoding="utf-8-sig")

Bersihkan quote luar tiap baris

In [37]:
# clean_lines = []

# for line in text.splitlines():
#     line = line.strip()
    
#     if line.startswith('"') and line.endswith('"'):
#         line = line[1:-1]
    
#     line = line.replace('""', '"')
#     clean_lines.append(line)

# clean_text = "\n".join(clean_lines)


Load clean data

In [3]:
# 3. Load ulang sebagai CSV normal
df = pd.read_csv(file_path)

df.head()

,Date,Price,Open,High,Low,Vol.,Change %
0,05/11/2026,"20,514.2","20,457.7","20,522.9","20,400.4",NaN,0.23%
1,05/08/2026,"20,466.4","20,327.0","20,469.9","20,320.1",NaN,0.69%
2,05/07/2026,"20,326.2","20,427.4","20,451.7","20,321.0",NaN,-0.48%
3,05/06/2026,"20,423.9","20,363.4","20,509.1","20,349.7",NaN,0.30%
4,05/05/2026,"20,363.4","20,317.7","20,402.1","20,287.3",NaN,0.27%


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180 entries, 0 to 179
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Date      180 non-null    object 
 1   Price     180 non-null    object 
 2   Open      180 non-null    object 
 3   High      180 non-null    object 
 4   Low       180 non-null    object 
 5   Vol.      0 non-null      float64
 6   Change %  180 non-null    object 
dtypes: float64(1), object(6)
memory usage: 10.0+ KB


# Data preprocessing

## Rename column

In [5]:
df = df.rename(columns=
{
    "Date": "timestamp",
    "Price": "close",
    "Open": "open",
    "High": "high",
    "Low": "low",
    "Vol.": "volume"
})

In [6]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180 entries, 0 to 179
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   timestamp  180 non-null    object 
 1   close      180 non-null    object 
 2   open       180 non-null    object 
 3   high       180 non-null    object 
 4   low        180 non-null    object 
 5   volume     0 non-null      float64
 6   Change %   180 non-null    object 
dtypes: float64(1), object(6)
memory usage: 10.0+ KB
None


## Bersihkan angka ribuan

In [7]:
price_cols = ["open", "high", "low", "close"]
for col in price_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(",", "", regex=False)
        .astype(float)
    )

## Convert timestamp

In [8]:
df["timestamp"] = pd.to_datetime(df["timestamp"])

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180 entries, 0 to 179
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   timestamp  180 non-null    datetime64[ns]
 1   close      180 non-null    float64       
 2   open       180 non-null    float64       
 3   high       180 non-null    float64       
 4   low        180 non-null    float64       
 5   volume     0 non-null      float64       
 6   Change %   180 non-null    object        
dtypes: datetime64[ns](1), float64(5), object(1)
memory usage: 10.0+ KB


## Isi dengan dummy data

In [10]:
# Volume tidak tersedia, isi dummy
df["volume"] = 0.0
df["amount"] = 0.0

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180 entries, 0 to 179
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   timestamp  180 non-null    datetime64[ns]
 1   close      180 non-null    float64       
 2   open       180 non-null    float64       
 3   high       180 non-null    float64       
 4   low        180 non-null    float64       
 5   volume     180 non-null    float64       
 6   Change %   180 non-null    object        
 7   amount     180 non-null    float64       
dtypes: datetime64[ns](1), float64(6), object(1)
memory usage: 11.4+ KB


## Sort lama ke baru

In [12]:
df = df.sort_values("timestamp").reset_index(drop=True)

In [13]:
df.head()

,timestamp,close,open,high,low,volume,Change %,amount
0,2025-09-02,19091.2,19223.6,19246.0,19043.7,0.0,-0.69%,0.0
1,2025-09-03,19141.8,19089.6,19177.9,19060.3,0.0,0.27%,0.0
2,2025-09-04,19126.8,19140.9,19187.0,19095.6,0.0,-0.08%,0.0
3,2025-09-05,19242.6,19126.8,19311.6,19122.7,0.0,0.61%,0.0
4,2025-09-08,19180.0,19242.6,19245.9,19106.7,0.0,-0.33%,0.0


## Ambil kolom untuk Kronos

In [14]:
kronos_df = df[["timestamp", "open", "high", "low", "close", "volume", "amount"]].copy()

In [15]:
kronos_df.head()

,timestamp,open,high,low,close,volume,amount
0,2025-09-02,19223.6,19246.0,19043.7,19091.2,0.0,0.0
1,2025-09-03,19089.6,19177.9,19060.3,19141.8,0.0,0.0
2,2025-09-04,19140.9,19187.0,19095.6,19126.8,0.0,0.0
3,2025-09-05,19126.8,19311.6,19122.7,19242.6,0.0,0.0
4,2025-09-08,19242.6,19245.9,19106.7,19180.0,0.0,0.0


## Hapus missing value kalau ada

In [16]:
kronos_df = kronos_df.dropna()
kronos_df.head()

,timestamp,open,high,low,close,volume,amount
0,2025-09-02,19223.6,19246.0,19043.7,19091.2,0.0,0.0
1,2025-09-03,19089.6,19177.9,19060.3,19141.8,0.0,0.0
2,2025-09-04,19140.9,19187.0,19095.6,19126.8,0.0,0.0
3,2025-09-05,19126.8,19311.6,19122.7,19242.6,0.0,0.0
4,2025-09-08,19242.6,19245.9,19106.7,19180.0,0.0,0.0


In [17]:
kronos_df.to_csv("./data/ground-truth/eur_idr_gt_6m.csv", index=False)